#**Project : Analyzing the trends of COVID-19 with Python**


##**Step 1:- Import Libraries**

In [370]:
!pip install prophet

In [371]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from prophet import Prophet

##**Step 2 :- Load Dataset**

In [397]:
df = pd.read_csv("/content/sample_data/covid_19_clean_complete.csv")
df

,Province/State,Country/Region,Lat,Long,Date,Confirmed,Deaths,Recovered,Active,WHO Region
0,NaN,Afghanistan,33.939110,67.709953,2020-01-22,0,0,0,0,Eastern Mediterranean
1,NaN,Albania,41.153300,20.168300,2020-01-22,0,0,0,0,Europe
2,NaN,Algeria,28.033900,1.659600,2020-01-22,0,0,0,0,Africa
3,NaN,Andorra,42.506300,1.521800,2020-01-22,0,0,0,0,Europe
4,NaN,Angola,-11.202700,17.873900,2020-01-22,0,0,0,0,Africa
...,...,...,...,...,...,...,...,...,...,...
49063,NaN,Sao Tome and Principe,0.186400,6.613100,2020-07-27,865,14,734,117,Africa
49064,NaN,Yemen,15.552727,48.516388,2020-07-27,1691,483,833,375,Eastern Mediterranean
49065,NaN,Comoros,-11.645500,43.333300,2020-07-27,354,7,328,19,Africa
49066,NaN,Tajikistan,38.861000,71.276100,2020-07-27,7235,60,6028,1147,Europe


In [373]:
df.head()

,Province/State,Country/Region,Lat,Long,Date,Confirmed,Deaths,Recovered,Active,WHO Region
0,NaN,Afghanistan,33.93911,67.709953,2020-01-22,0,0,0,0,Eastern Mediterranean
1,NaN,Albania,41.15330,20.168300,2020-01-22,0,0,0,0,Europe
2,NaN,Algeria,28.03390,1.659600,2020-01-22,0,0,0,0,Africa
3,NaN,Andorra,42.50630,1.521800,2020-01-22,0,0,0,0,Europe
4,NaN,Angola,-11.20270,17.873900,2020-01-22,0,0,0,0,Africa


In [374]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49068 entries, 0 to 49067
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Province/State  14664 non-null  object 
 1   Country/Region  49068 non-null  object 
 2   Lat             49068 non-null  float64
 3   Long            49068 non-null  float64
 4   Date            49068 non-null  object 
 5   Confirmed       49068 non-null  int64  
 6   Deaths          49068 non-null  int64  
 7   Recovered       49068 non-null  int64  
 8   Active          49068 non-null  int64  
 9   WHO Region      49068 non-null  object 
dtypes: float64(2), int64(4), object(4)
memory usage: 3.7+ MB


In [375]:
df.shape

(49068, 10)

##**Step 3 :- Data Cleaning**

In [398]:
df.rename(columns={"Province/State":"State","Country/Region":"Country"},inplace=True)
df


,State,Country,Lat,Long,Date,Confirmed,Deaths,Recovered,Active,WHO Region
0,NaN,Afghanistan,33.939110,67.709953,2020-01-22,0,0,0,0,Eastern Mediterranean
1,NaN,Albania,41.153300,20.168300,2020-01-22,0,0,0,0,Europe
2,NaN,Algeria,28.033900,1.659600,2020-01-22,0,0,0,0,Africa
3,NaN,Andorra,42.506300,1.521800,2020-01-22,0,0,0,0,Europe
4,NaN,Angola,-11.202700,17.873900,2020-01-22,0,0,0,0,Africa
...,...,...,...,...,...,...,...,...,...,...
49063,NaN,Sao Tome and Principe,0.186400,6.613100,2020-07-27,865,14,734,117,Africa
49064,NaN,Yemen,15.552727,48.516388,2020-07-27,1691,483,833,375,Eastern Mediterranean
49065,NaN,Comoros,-11.645500,43.333300,2020-07-27,354,7,328,19,Africa
49066,NaN,Tajikistan,38.861000,71.276100,2020-07-27,7235,60,6028,1147,Europe


In [377]:
# Convert date
df['Date'] = pd.to_datetime(df['Date'])

In [400]:
df_global = df.groupby('Date')[['Confirmed','Recovered','Deaths']].sum().reset_index()
df_global

,Date,Confirmed,Recovered,Deaths
0,2020-01-22,555,28,17
1,2020-01-23,654,30,18
2,2020-01-24,941,36,26
3,2020-01-25,1434,39,42
4,2020-01-26,2118,52,56
...,...,...,...,...
183,2020-07-23,15510481,8710969,633506
184,2020-07-24,15791645,8939705,639650
185,2020-07-25,16047190,9158743,644517
186,2020-07-26,16251796,9293464,648621


In [399]:
df_country = df.groupby(['Country'])[['Confirmed','Recovered','Deaths']].sum().reset_index()
df_country

,Country,Confirmed,Recovered,Deaths
0,Afghanistan,1936390,798240,49098
1,Albania,196702,118877,5708
2,Algeria,1179755,755897,77972
3,Andorra,94404,69074,5423
4,Angola,22662,6573,1078
...,...,...,...,...
182,West Bank and Gaza,233461,61124,1370
183,Western Sahara,901,648,63
184,Yemen,67180,23779,17707
185,Zambia,129421,83611,2643


In [379]:
# Sort Values by Date
df_global.sort_values('Date')

,Date,Confirmed,Recovered,Deaths
0,2020-01-22,555,28,17
1,2020-01-23,654,30,18
2,2020-01-24,941,36,26
3,2020-01-25,1434,39,42
4,2020-01-26,2118,52,56
...,...,...,...,...
183,2020-07-23,15510481,8710969,633506
184,2020-07-24,15791645,8939705,639650
185,2020-07-25,16047190,9158743,644517
186,2020-07-26,16251796,9293464,648621


##**Step 4:- Basic Data Understanding (EDA)**

In [380]:
# Total Global Impact
df_global[['Confirmed','Deaths','Recovered']].sum()

,0
Confirmed,828508482
Deaths,43384903
Recovered,388408229


In [381]:
# Latest Data
latest = df_global[df_global['Date'] == df_global['Date'].max()]
latest.head(10)

,Date,Confirmed,Recovered,Deaths
187,2020-07-27,16480485,9468087,654036


##**Step 5:- Country wise Analysis**

In [402]:
top_countries = df_country.sort_values('Confirmed',ascending=False).head(10)
top_countries

,Country,Confirmed,Recovered,Deaths
173,US,224345948,56353416,11011411
23,Brazil,89524967,54492873,3938034
138,Russia,45408411,25120448,619385
79,India,40883464,23783720,1111831
157,Spain,27404045,15093583,3033030
177,United Kingdom,26748587,126217,3997775
85,Italy,26745145,15673910,3707717
61,France,21210926,7182115,3048524
65,Germany,21059152,17107839,871322
81,Iran,19339267,15200895,1024136


In [403]:
fig = px.bar(top_countries,x='Country',y='Confirmed',color='Country',template='plotly_dark',title='Top 10 Countries with Highest Infection Rate')
fig.show()

##**Step 6:- Feature Engineering**

In [382]:
# Daily New Cases
df_global['New_Cases'] = df_global['Confirmed'].diff()
df_global['New_Cases']

,New_Cases
0,NaN
1,99.0
2,287.0
3,493.0
4,684.0
...,...
183,282756.0
184,281164.0
185,255545.0
186,204606.0


In [383]:
# Recovery Rate
df_global['Recovery_Rate'] = df_global['Recovered'] / df_global['Confirmed']
df_global['Recovery_Rate']

,Recovery_Rate
0,0.050450
1,0.045872
2,0.038257
3,0.027197
4,0.024551
...,...
183,0.561618
184,0.566103
185,0.570738
186,0.571842


In [384]:
# Death Rate
df_global['Death_Rate'] = df_global['Deaths'] / df_global['Confirmed']
df_global['Death_Rate']

,Death_Rate
0,0.030631
1,0.027523
2,0.027630
3,0.029289
4,0.026440
...,...
183,0.040844
184,0.040506
185,0.040164
186,0.039911


##**Step 7:- Time Trend Visualization**

In [385]:
# Confirmed Cases
fig = px.line(df_global,x='Date',y='Confirmed',template='plotly_dark',title='Confirmed Cases Over Time')
fig.show()

In [386]:
# Infection Rate
fig = px.line(df_global,x='Date',y='New_Cases',template='plotly_dark',title='Daily Infection Rate')
fig.show()

In [387]:
# Recovery Rate
fig = px.line(df_global,x='Date',y='Recovery_Rate',template='plotly_dark',title='Recovery Rate')
fig.show()

In [388]:
# Death Rate
fig = px.line(df_global,x='Date',y='Death_Rate',template='plotly_dark',title='Death Rate')
fig.show()

##**Step 8:- Moving Average**

In [389]:
df_global['MA_7'] = df_global['Confirmed'].rolling(7).mean()
df_global['MA_7']

,MA_7
0,NaN
1,NaN
2,NaN
3,NaN
4,NaN
...,...
183,1.475036e+07
184,1.499851e+07
185,1.524923e+07
186,1.549851e+07


In [390]:
# Plot
fig = px.line(df_global,x='Date',y=['Confirmed','MA_7'],template='plotly_dark',title='Trend with Moving Average')
fig.show()

##**Step 9:- Prepare Data for Prophet**

In [391]:
prophet_df = df_global[['Date','Confirmed']]
prophet_df.columns = ['ds','y']
prophet_df

,ds,y
0,2020-01-22,555
1,2020-01-23,654
2,2020-01-24,941
3,2020-01-25,1434
4,2020-01-26,2118
...,...,...
183,2020-07-23,15510481
184,2020-07-24,15791645
185,2020-07-25,16047190
186,2020-07-26,16251796


##**Step 10:- Train Model**

In [392]:
model = Prophet()
model.fit(prophet_df)

INFO:prophet:Disabling yearly seasonality. Run prophet with yearly_seasonality=True to override this.
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


##**Step 11:- Predict Future (7 Days)**

In [393]:
future = model.make_future_dataframe(periods=7)
forecast = model.predict(future)
forecast

,ds,trend,yhat_lower,yhat_upper,trend_lower,trend_upper,additive_terms,additive_terms_lower,additive_terms_upper,weekly,weekly_lower,weekly_upper,multiplicative_terms,multiplicative_terms_lower,multiplicative_terms_upper,yhat
0,2020-01-22,-9.613288e+03,-1.262210e+05,8.227921e+04,-9.613288e+03,-9.613288e+03,-11063.558307,-11063.558307,-11063.558307,-11063.558307,-11063.558307,-11063.558307,0.0,0.0,0.0,-2.067685e+04
1,2020-01-23,-6.933409e+03,-1.187508e+05,9.364858e+04,-6.933409e+03,-6.933409e+03,-1117.543863,-1117.543863,-1117.543863,-1117.543863,-1117.543863,-1117.543863,0.0,0.0,0.0,-8.050953e+03
2,2020-01-24,-4.253530e+03,-9.871670e+04,1.015614e+05,-4.253530e+03,-4.253530e+03,10080.978737,10080.978737,10080.978737,10080.978737,10080.978737,10080.978737,0.0,0.0,0.0,5.827449e+03
3,2020-01-25,-1.573651e+03,-9.292420e+04,1.235270e+05,-1.573651e+03,-1.573651e+03,13750.326871,13750.326871,13750.326871,13750.326871,13750.326871,13750.326871,0.0,0.0,0.0,1.217668e+04
4,2020-01-26,1.106228e+03,-9.545570e+04,1.200089e+05,1.106228e+03,1.106228e+03,7298.791978,7298.791978,7298.791978,7298.791978,7298.791978,7298.791978,0.0,0.0,0.0,8.405020e+03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
190,2020-07-30,1.674503e+07,1.663289e+07,1.685423e+07,1.673900e+07,1.675107e+07,-1117.543863,-1117.543863,-1117.543863,-1117.543863,-1117.543863,-1117.543863,0.0,0.0,0.0,1.674392e+07
191,2020-07-31,1.694902e+07,1.684700e+07,1.706682e+07,1.693740e+07,1.695928e+07,10080.978737,10080.978737,10080.978737,10080.978737,10080.978737,10080.978737,0.0,0.0,0.0,1.695911e+07
192,2020-08-01,1.715301e+07,1.705215e+07,1.727908e+07,1.713494e+07,1.717175e+07,13750.326871,13750.326871,13750.326871,13750.326871,13750.326871,13750.326871,0.0,0.0,0.0,1.716677e+07
193,2020-08-02,1.735701e+07,1.725300e+07,1.747153e+07,1.732977e+07,1.738529e+07,7298.791978,7298.791978,7298.791978,7298.791978,7298.791978,7298.791978,0.0,0.0,0.0,1.736430e+07


##**Step 12:- Visualize Prediction**

In [394]:
fig = px.line(forecast,x='ds',y='yhat',template='plotly_dark',title='Predicted Cases')
fig.show()

##**Step 13:- Actual vs Predicted**

In [396]:
fig = go.Figure()

fig.add_trace(go.Scatter(x=df_global['Date'], y=df_global['Confirmed'], name='Actual'))
fig.add_trace(go.Scatter(x=forecast['ds'], y=forecast['yhat'], name='Predicted'))
fig.update_layout(template='plotly_dark')

fig.show()

#**Conclusion**

- COVID-19 cases showed exponential growth in early stages
- Recovery rate improved over time
- Death rate gradually declined
- Moving average helped identify actual trends
- Prophet model predicted short-term future cases

This analysis helps understand pandemic trends and supports decision-making.